In [ ]:
import pandas as pd
from pathlib import Path

source_path = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\Moca\2026\moca_montreal_cognitive_assessment.tsv.xlsx")
all_output_path = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\Moca\2026\moca_wide_all_participants.xlsx")
filtered_output_path = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\Moca\2026\moca_filtered_wide.xlsx")

# Long MoCA table.
moca_long = pd.read_excel(source_path).copy()
moca_long["id_participant"] = pd.to_numeric(moca_long["id_participant"], errors="coerce").astype("Int64")

# Map long timepoints to wide column labels.
tp_map = {
    "t00": "Initiale",
    "t02": "suivi-2ans",
    "t04": "suivi-4ans",
    "t06": "suivi-6ans",
    "t08": "suivi-8ans",
    "t10": "suivi-10ans",
}

moca_long["tp_label"] = moca_long["timepoint"].map(tp_map)

# Build the wide table from the complete long table first.
moca_wide_all = (
    moca_long
    .pivot_table(
        index="id_participant",
        columns="tp_label",
        values=[
            "age",
            "moca_score_total_30",
            "moca_score_total_plus_scolarite_30",
        ],
        aggfunc="first",
    )
)

moca_wide_all.columns = [f"{var}_{tp}" for var, tp in moca_wide_all.columns]
moca_wide_all = moca_wide_all.reset_index()

# Keep the full long-file participant set by default.
# Replace keep_ids with a separate cohort table if you want a narrower filter.
keep_ids = pd.Index(moca_wide_all["id_participant"].dropna().drop_duplicates(), name="id_participant")

# Save the full wide table first.
moca_wide_all.to_excel(all_output_path, index=False)

# Then filter only after the wide table exists.
moca_filtered_wide = moca_wide_all[moca_wide_all["id_participant"].isin(keep_ids)].copy()
moca_filtered_wide.to_excel(filtered_output_path, index=False)

moca_wide_all


,id_participant,age_Initiale,age_suivi-10ans,age_suivi-2ans,age_suivi-4ans,age_suivi-6ans,age_suivi-8ans,moca_score_total_30_Initiale,moca_score_total_30_suivi-10ans,moca_score_total_30_suivi-2ans,moca_score_total_30_suivi-4ans,moca_score_total_30_suivi-6ans,moca_score_total_30_suivi-8ans,moca_score_total_plus_scolarite_30_Initiale,moca_score_total_plus_scolarite_30_suivi-10ans,moca_score_total_plus_scolarite_30_suivi-2ans,moca_score_total_plus_scolarite_30_suivi-4ans,moca_score_total_plus_scolarite_30_suivi-6ans,moca_score_total_plus_scolarite_30_suivi-8ans
0,3002498,68.3,78.5,70.6,72.6,74.6,76.5,30.0,28.0,27.0,30.0,29.0,28.0,30.0,28.0,27.0,30.0,29.0,28.0
1,3004865,69.1,NaN,NaN,NaN,NaN,NaN,25.0,NaN,NaN,NaN,NaN,NaN,25.0,NaN,NaN,NaN,NaN,NaN
2,3014036,72.6,NaN,74.7,77.0,79.3,NaN,26.0,NaN,24.0,26.0,27.0,NaN,26.0,NaN,25.0,27.0,28.0,NaN
3,3025432,69.2,79.2,71.6,73.2,75.4,77.6,28.0,27.0,29.0,27.0,29.0,27.0,29.0,28.0,30.0,28.0,30.0,28.0
4,3029434,73.5,NaN,75.5,77.4,79.6,NaN,22.0,NaN,23.0,22.0,18.0,NaN,22.0,NaN,23.0,22.0,18.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
418,9571095,81.4,NaN,83.6,85.8,87.4,89.1,29.0,NaN,28.0,27.0,26.0,27.0,29.0,NaN,28.0,27.0,26.0,27.0
419,9648306,82.1,NaN,NaN,86.5,NaN,NaN,23.0,NaN,NaN,17.0,NaN,NaN,23.0,NaN,NaN,17.0,NaN,NaN
420,9889496,66.7,NaN,69.0,NaN,NaN,NaN,28.0,NaN,28.0,NaN,NaN,NaN,28.0,NaN,28.0,NaN,NaN,NaN
421,9928452,76.1,NaN,NaN,NaN,NaN,NaN,18.0,NaN,NaN,NaN,NaN,NaN,19.0,NaN,NaN,NaN,NaN,NaN


In [4]:
# --- Filter wide table using PSCIDs from presence_table.csv ---
from pathlib import Path

ids_path = Path(r"D:\04-Raw_fMRI\presence_table.csv")
# Read the presence table and try common column names; fallback to first column
ids_df = pd.read_csv(ids_path)
col_candidates = ['PSCID','pscid','id_participant','id','participant','participant_id']
keep_ids_list = None
for c in col_candidates:
    if c in ids_df.columns:
        keep_ids_list = ids_df[c].dropna().astype(int).unique().tolist()
        break
if keep_ids_list is None:
    keep_ids_list = ids_df.iloc[:,0].dropna().astype(int).unique().tolist()

# Ensure `moca_wide_all` exists (built in prior cell).
try:
    _ = moca_wide_all
except NameError:
    raise RuntimeError('Run the wide-pivot cell first to produce `moca_wide_all`.')

# Normalize types and filter
moca_wide_all['id_participant'] = pd.to_numeric(moca_wide_all['id_participant'], errors='coerce').astype('Int64')
keep_ids = pd.Index(pd.to_numeric(pd.Series(keep_ids_list), errors='coerce').dropna().astype('Int64').unique())

moca_wide_selected = moca_wide_all[moca_wide_all['id_participant'].isin(keep_ids)].copy()

print(f'Selected rows: {len(moca_wide_selected)}')
moca_wide_selected.head(10)

# Save filtered wide table
output_path = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\Moca\2026\moca_filtered_wide_selected.xlsx")
moca_wide_selected.to_excel(output_path, index=False)
print('Saved filtered wide table to:', output_path)


KeyError: 'PSCID'

In [ ]:
# --- Create moca_long_107.xlsx by filtering the long source table with PSCIDs from the wide 107 file ---
from pathlib import Path
import pandas as pd

source_path = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\Moca\2026\moca_montreal_cognitive_assessment.tsv.xlsx")
keep_path = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\Moca\2026\moca_filtered_wide_107.xlsx")
output_path = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\Moca\2026\moca_long_107.xlsx")

source_df = pd.read_excel(source_path)
keep_df = pd.read_excel(keep_path)

id_candidates = ['PSCID', 'pscid', 'id_participant', 'participant', 'participant_id', 'id']
keep_ids = None
for col in id_candidates:
    if col in keep_df.columns:
        keep_ids = pd.to_numeric(keep_df[col], errors='coerce').dropna().astype('Int64').unique()
        break
if keep_ids is None:
    keep_ids = pd.to_numeric(keep_df.iloc[:, 0], errors='coerce').dropna().astype('Int64').unique()

source_id_candidates = ['PSCID', 'pscid', 'id_participant', 'participant', 'participant_id', 'id']
source_id_col = None
for col in source_id_candidates:
    if col in source_df.columns:
        source_id_col = col
        break
if source_id_col is None:
    raise KeyError(f'No participant id column found in source table. Available columns: {list(source_df.columns)}')

source_df[source_id_col] = pd.to_numeric(source_df[source_id_col], errors='coerce').astype('Int64')
filtered_df = source_df[source_df[source_id_col].isin(pd.Index(keep_ids))].copy()

filtered_df.to_excel(output_path, index=False)
print(f'Saved {len(filtered_df)} rows to {output_path}')
filtered_df

KeyError: 'PSCID'